# Imports

In [0]:
from datetime import datetime, timedelta

# Parâmetros do notebook

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
print("Environment:", environment)

In [0]:
dbutils.widgets.text("id_projeto", "endometriose", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.now().strftime("%Y-%m-%d")
print(f"Data Execução Modelo: {data_execucao_modelo}")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

print(f"environment_tbl: {environment_tbl}")

In [0]:
root_path_work = f"/mnt/trusted/datalake/ia/data/{id_projeto}/work/"
print(f"root_path_work: {root_path_work}")

In [0]:
# dbutils.fs.ls(root_path_work)


# Funções Auxiliares

In [0]:
# tables = dbutils.fs.ls(root_path_work)

# for table in tables:
#     table_name = table.name.replace("/", "")
#     if f"tbl_wrk_{id_projeto}" in table_name:
#         print(f"Removendo: {table_name} - {table.path}")
#         spark.sql(f"drop table if exists ia.{table_name}")
#         dbutils.fs.rm(table.path, recurse=True)

In [0]:
def optimize_table(table_id):
    spark.sql(f"VACUUM {table_id}")
    spark.sql(f"OPTIMIZE {table_id}")
    spark.sql(f"ANALYZE TABLE {table_id} COMPUTE STATISTICS")

# Tabelas Work

## Variáveis com os nomes das tabelas

In [0]:
table_name_wrk_exame = f"{environment_tbl}tbl_wrk_{id_projeto}_exame"
table_name_wrk_laudo = f"{environment_tbl}tbl_wrk_{id_projeto}_laudo"
table_name_wrk_convenio = f"{environment_tbl}tbl_wrk_{id_projeto}_convenio"
table_name_wrk_paciente = f"{environment_tbl}tbl_wrk_{id_projeto}_paciente"

view_name_entrada = f"{environment_tbl}vw_{id_projeto}_entrada"
table_name_entrada = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_entrada"

In [0]:
print(f"{'table_name_wrk_exame':40}: {table_name_wrk_exame}")
print(f"{'table_name_wrk_laudo':40}: {table_name_wrk_laudo}")
print(f"{'table_name_wrk_convenio':40}: {table_name_wrk_convenio}")
print(f"{'table_name_wrk_paciente':40}: {table_name_wrk_paciente}")
print(f"{'view_name_entrada':40}: {view_name_entrada}")
print(f"{'table_name_entrada':40}: {table_name_entrada}")

In [0]:
# df = spark.sql(f"SELECT COUNT(1) FROM ia.{table_name_entrada}")
# count = df.collect()[0][0]

# if count > 0:
#     days = 30
# else:
#     days = 30

days = 50
current_date = datetime.strptime(data_execucao_modelo, "%Y-%m-%d").date()
initial_date = (current_date - timedelta(days=days)).strftime("%Y-%m-%d")

print(f"initial_date: {initial_date}")

## tbl_work_exame

In [0]:
table_name_wrk_exame

In [0]:
spark.sql(f"""
    create or replace table ia.{table_name_wrk_exame}
    location '{root_path_work}{table_name_wrk_exame}'
    with cte as (
      select
        idExame,
        dataExame,
        dataLiberacaoLaudo,
        tipoCodigo,
        codigo,
        nomeCodigo,
        if(lower(trim(descricaoProcedimento)) != "none", descricaoProcedimento, null) as descricaoProcedimento,

        idOrganization as idUnidade,
        descricaoCorporativaUnidade as nomeUnidade,
        cnpjHospital as cnpjUnidade,
        nomeRegionalHospital as regionalUnidade,
        tipoIntegracao as tipoUnidade,

        idMedico,
        numCrm,
        ufCrm,
        medicoEncaminhador as nomeMedico,

        idPatient as idPaciente,

        translate(
            exame.descricaoProcedimento,
            'ÀÁÂÃÄÅàáâãäåÇçÈÉÊËèéêëÌÍÎÏìíîïÒÓÔÕÖòóôõöÙÚÛÜùúûüÝý',
            'AAAAAAaaaaaaCcEEEEeeeeIIIIiiiiOOOOOoooooUUUUuuuuYy'
        ) as descricaoProcedimentoLimpo,

        listaExames,
        idEncounter,
        nomeConvenio

      from ia.tbl_gold_exame as exame

      where true
        and date(exame.dataExame) >= '{initial_date}'
        and date(exame.dataExame) <= '{data_execucao_modelo}'

        -- Fixando para não ler registros anteriores a essa data, pode ser removido no futuro!
        and date(exame.dataExame) >= '2024-06-01'
      
      qualify row_number() over (partition by idExame order by dataParticao desc) = 1
    )
    select * from cte
    where (descricaoProcedimentoLimpo ilike '%abdome%'
    or descricaoProcedimentoLimpo ilike '%abidome%'
    or descricaoProcedimentoLimpo ilike '%pelve%'
    or descricaoProcedimentoLimpo ilike '%peuve%'
    or descricaoProcedimentoLimpo like '%bexiga%'
    or descricaoProcedimentoLimpo like '%bixiga%'
    or descricaoProcedimentoLimpo like '%utero%'
    or descricaoProcedimentoLimpo like '%via%urinaria%')

""").display()

In [0]:
optimize_table(f"ia.{table_name_wrk_exame}")

In [0]:
spark.sql(f"select count(1) as qtd, min(dataExame), max(dataExame) from ia.{table_name_wrk_exame}").display()

## tbl_wrk_laudo

In [0]:
table_name_wrk_laudo

In [0]:
spark.sql(f"""
    create or replace table ia.{table_name_wrk_laudo}
    location '{root_path_work}{table_name_wrk_laudo}'
    as
    with cte as (
        select
            idExame,
            explode(listaExames) as exames
        from ia.{table_name_wrk_exame} as laudo
    )
    select
        idExame,
        trim(exames.laudoOriginal) as laudoOriginal
    from cte
    where ifnull(exames.laudoOriginal, "") != ""
      and trim(lower(exames.laudoOriginal)) != "none"
    qualify
        row_number() over(
            partition by
                idExame
            order by
                exames.laudoOriginal,
                exames.nomeExame
        ) = 1
""").display()

In [0]:
optimize_table(f"ia.{table_name_wrk_laudo}")

In [0]:
spark.sql(f"select count(1) as qtd from ia.{table_name_wrk_laudo}").display()

## tbl_wrk_convenio

In [0]:
table_name_wrk_convenio

In [0]:
spark.sql(f"""
    create or replace table ia.{table_name_wrk_convenio}
    location '{root_path_work}{table_name_wrk_convenio}'
    as
    with cte as (
        select distinct
            convenio.idExame,
            ifnull(convenio.nomeConvenio, enc.extension_nome_local_convenio) as nomeConvenio

        from ia.{table_name_wrk_exame} as convenio

        left join ia.tbl_silver_encounter as enc
            on convenio.idEncounter = enc.idEncounter
    )
    select * from cte
    where nomeConvenio is not null
""").display()

In [0]:
optimize_table(f"ia.{table_name_wrk_convenio}")

In [0]:
spark.sql(f"select count(1) as qtd from ia.{table_name_wrk_convenio}").display()

## tbl_wrk_paciente

In [0]:
table_name_wrk_paciente

In [0]:
spark.sql(f"""
    create or replace table ia.{table_name_wrk_paciente}
    location '{root_path_work}{table_name_wrk_paciente}'
    as
    with cte_ids as (
        select distinct idPaciente
        from ia.{table_name_wrk_exame}
    )
    ,cte_tel as (
    select
        tel.idPatient as idPaciente,
        coalesce(concat(tel.telecom_ddd, tel.telecom_number),  tel.telecom_value) as telefoneContato
    from ia.tbl_silver_patient_telecom as tel
    inner join cte_ids as ids
        on tel.idPatient = ids.idPaciente
    where lower(trim(tel.telecom_system)) = 'phone'
    qualify row_number() over (partition by tel.idPatient order by tel.telecom_updateDate desc) = 1
    )
    select
        paciente.idPatient as idPaciente,
        paciente.numCpf as cpfPaciente,
        paciente.nomePaciente,
        paciente.dataNascimento as dataNascimentoPaciente,
        paciente.idadePaciente,
        case lower(trim(paciente.nomeGenero))
            when 'male' then 'M'
            when 'female' then 'F'
            else '-'
        end as sexoPaciente,
        tel.telefoneContato
    from ia.tbl_gold_paciente as paciente
    inner join cte_ids as ids
        on paciente.idPatient = ids.idPaciente
    left join cte_tel as tel
        on paciente.idPatient = tel.idPaciente
""").display()

In [0]:
optimize_table(f"ia.{table_name_wrk_paciente}")

In [0]:
spark.sql(f"select count(1) as qtd from ia.{table_name_wrk_paciente}").display()

# View Entrada Modelo

In [0]:
view_name_entrada

In [0]:
spark.sql(f"""
    create or replace temporary view {view_name_entrada}
    as
    select
        exame.idExame,
        exame.dataExame,
        exame.dataLiberacaoLaudo,
        convenio.nomeConvenio,
        exame.tipoCodigo,
        exame.codigo,
        exame.nomeCodigo,
        exame.descricaoProcedimento,
        exame.idUnidade,
        exame.nomeUnidade,
        exame.cnpjUnidade,
        exame.regionalUnidade,
        exame.tipoUnidade,
        exame.idMedico,
        exame.numCrm,
        exame.ufCrm,
        exame.nomeMedico,
        exame.idPaciente,
        paciente.cpfPaciente,
        paciente.nomePaciente,
        paciente.dataNascimentoPaciente,
        paciente.idadePaciente,
        paciente.sexoPaciente,
        paciente.telefoneContato,
        laudo.laudoOriginal,

        row_number() over(
            partition by
                exame.idPaciente,
                date(exame.dataLiberacaoLaudo),
                laudo.laudoOriginal
            order by
                exame.dataLiberacaoLaudo,
                exame.descricaoProcedimento,
                exame.idExame
        ) as laudoDuplicado

    from ia.{table_name_wrk_exame} as exame

    inner join ia.{table_name_wrk_laudo} as laudo
        on exame.idExame = laudo.idExame

    left join ia.{table_name_wrk_convenio} as convenio
        on exame.idExame = convenio.idExame

    inner join ia.{table_name_wrk_paciente} as paciente
        on exame.idPaciente = paciente.idPaciente

""").display()  

In [0]:
spark.sql(f"select count(1) as qtd from {view_name_entrada}").display()

# Carrega Tabela Entrada

In [0]:
table_name_entrada

In [0]:
spark.sql(f"""
    delete from ia.{table_name_entrada}
    where dataExecucaoModelo = '{data_execucao_modelo}'
""").display()

In [0]:
print(view_name_entrada)
print(table_name_entrada)

In [0]:
spark.sql(f"""
    insert into ia.{table_name_entrada}
    (
        dataExecucaoModelo,
        idPredicao,

        idExame,
        dataExame,
        dataLiberacaoLaudo,
        nomeConvenio,
        tipoCodigo,
        codigo,
        nomeCodigo,
        descricaoProcedimento,
        idUnidade,
        nomeUnidade,
        cnpjUnidade,
        regionalUnidade,
        tipoUnidade,
        idMedico,
        numCrm,
        ufCrm,
        nomeMedico,
        idPaciente,
        cpfPaciente,
        nomePaciente,
        dataNascimentoPaciente,
        idadePaciente,
        sexoPaciente,
        telefoneContato,
        laudoOriginal,
        laudoDuplicado,

        dataCarga
    )
    select
        date('{data_execucao_modelo}') as dataExecucaoModelo,
        uuid() as idPredicao,

        idExame,
        dataExame,
        dataLiberacaoLaudo,
        nomeConvenio,
        tipoCodigo,
        codigo,
        nomeCodigo,
        descricaoProcedimento,
        idUnidade,
        nomeUnidade,
        cnpjUnidade,
        regionalUnidade,
        tipoUnidade,
        idMedico,
        numCrm,
        ufCrm,
        nomeMedico,
        idPaciente,
        cpfPaciente,
        nomePaciente,
        dataNascimentoPaciente,
        idadePaciente,
        sexoPaciente,
        telefoneContato,
        laudoOriginal,
        laudoDuplicado,

        current_timestamp() as dataCarga

    from {view_name_entrada}

    where idExame not in (select idExame from ia.{table_name_entrada})
""").display()

In [0]:
optimize_table(f"ia.{table_name_entrada}")

In [0]:
spark.sql(f"""
    select idExame, count(*) as qtd
    from ia.{table_name_entrada}
    group by all
    having qtd > 1
""").display()

In [0]:
spark.sql(f"""
    select
        dataExecucaoModelo,
        count(*) as qtd,
        sum(if(laudoDuplicado = 1, 1, 0)) as qtdLaudoUnico,
        min(dataExame) as minDataExame, max(dataExame) as maxDataExame
    from ia.{table_name_entrada}
    group by all
    order by dataExecucaoModelo desc
""").display()